In [0]:
watermark = spark.sql("""

SELECT watermark

FROM metadata.pipeline_metadata

WHERE pipeline_name =
'sales_pipeline'

""").collect()[0][0]

print(watermark)

2026-06-18 17:16:29.779246


In [0]:
jdbc_url = (
    "jdbc:postgresql://**********ooler.c-9.us-east-1.aws.neon.tech:5432/neondb?sslmode=require"
)
user = "neondb_owner"

password = "*****"

In [0]:
df_incremental = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        f"""
        SELECT *
        FROM sales
        WHERE updated_at >
        '{watermark}'
        """
    )
    .option("user", user)
    .option("password", password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(df_incremental)

customer_id,amount,region,updated_at


In [0]:
record_count = df_incremental.count()

if record_count == 0:

    print("No new records found.")
    spark.sql(
        "TRUNCATE TABLE staging.sales_incremental"
    )
    
else:

    from pyspark.sql.functions import current_timestamp

    df_incremental = df_incremental.withColumn(
        "ingestion_time",
        current_timestamp()
    )

    df_incremental.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("bronze.sales")

    df_incremental.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("staging.sales_incremental")

    print("Incremental Load Successful")

No new records found.
